# SE374.R11 - Multi-Layer CAM Generation, Affinity Matting & Stage 2 DeepLabV3+ Training
## Production-Grade Colab Pro Notebook: Pseudo-Supervision from Scratch
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khoinguyen59/medical-wsss/blob/master/notebooks/02_Pseudo_Masks_Train_Stage2_T4.ipynb)


* **Course:** SE374.R11 - Phat Trien Phan Mem Xu Ly Anh Y Khoa (UIT - DHQG-HCM)
* **Supervising Faculty:** TS. Bui Cao Doanh
* **Task:** Generate multi-layer Class Activation Maps (CAMs) with GPU Affinity Matting & Otsu clinical gating, then train DeepLabV3+ ResNet-101 (59M parameters) from scratch.
* **Dedicated Hardware:** NVIDIA Tesla T4 GPU (16GB VRAM, Turing Tensor Cores, Mixed Precision FP16)
* **Loss Formulation:** Multi-Layer Pseudo-Supervision (MLPS) Loss combining `PM_bn7`, `PM_b5_2`, `PM_b4_5` with Boundary Uncertainty Regularization & Classification Gate Mechanism (CGM).
* **Checkpoints:** Direct high-speed NVMe SSD storage on Colab VM disk (`/content/checkpoints/`) with instant one-click download (Zero Google Drive dependency).


---
## Step 1: Configure High-Speed Colab Local Storage (/content/checkpoints/) - Zero Drive Required
Configures high-speed local NVMe SSD storage at `/content/checkpoints/` directly on the Colab VM disk.


In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Pure Colab Local NVMe Storage (Pure GitHub + Colab, Zero Drive Popup)
if IN_COLAB:
    CHECKPOINT_DIR = Path("/content/checkpoints")
    print("[STORAGE] Using high-speed Colab VM local disk at /content/checkpoints")
else:
    CHECKPOINT_DIR = Path("./checkpoints")

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
STAGE1_BEST_CKPT = CHECKPOINT_DIR / "stage1_retrained_best.pth"
STAGE2_LATEST_CKPT = CHECKPOINT_DIR / "stage2_latest.pth"
STAGE2_BEST_CKPT = CHECKPOINT_DIR / "stage2_retrained_best.pth"
print(f"[CONFIG] Checkpoint directory: {CHECKPOINT_DIR.resolve()}")
print(f"[CONFIG] Stage 1 source checkpoint: {STAGE1_BEST_CKPT}")
print(f"[CONFIG] Stage 2 target checkpoint: {STAGE2_BEST_CKPT}")


---
## Step 2: Auto-Discover Codebase & Initialize GPU Environment
Discovers `src` module across Drive and local directories, falls back to GitHub clone.


In [ ]:
import os
import sys
from pathlib import Path
import random
import subprocess
import zipfile
import numpy as np
import torch

# Pure GitHub Codebase Discovery
CANDIDATES = [
    Path("/content/medical_wsss_repo"),
    Path("/content/medical-wsss"),
    Path.cwd(),
    Path.cwd().parent,
]

CODE_ROOT = next((str(p.resolve()) for p in CANDIDATES if (p / "src" / "models").is_dir()), None)
if CODE_ROOT is None:
    print("[GIT] Cloning official repository https://github.com/khoinguyen59/medical-wsss.git ...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/khoinguyen59/medical-wsss.git", "/content/medical_wsss_repo"], check=False)
    if Path("/content/medical_wsss_repo/src/models").is_dir():
        CODE_ROOT = str(Path("/content/medical_wsss_repo").resolve())
    else:
        CODE_ROOT = str(Path.cwd().resolve())

if CODE_ROOT in sys.path:
    sys.path.remove(CODE_ROOT)
sys.path.insert(0, CODE_ROOT)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.float16  # Native Turing Tensor Core FP16
NUM_WORKERS = 2
DRY_RUN = False  # Set to True for fast 1-epoch / smoke test
print(f"[READY] Code root: {CODE_ROOT}")
print(f"[READY] Device: {DEVICE} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}) [Tesla T4 Tier]")
print(f"[READY] Mixed Precision AMP Dtype: {AMP_DTYPE} (Tesla T4 Optimized)")
print(f"[READY] DRY_RUN: {DRY_RUN} | NUM_WORKERS: {NUM_WORKERS}")


---
## Step 3: Verify Dataset & Load Stage 1 Classifier Checkpoint
Loads Stage 1 retrained checkpoint or author baseline (`stage1_checkpoint_trained_on_bcss.pth`).


In [ ]:
import os
import sys
import shutil
import zipfile
import urllib.request
from pathlib import Path
import torch

IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'DEVICE' not in globals() else DEVICE
CHECKPOINT_DIR = Path('/content/checkpoints') if IN_COLAB else Path('./checkpoints')
STAGE1_BEST_CKPT = CHECKPOINT_DIR / 'stage1_retrained_best.pth' if 'STAGE1_BEST_CKPT' not in globals() else STAGE1_BEST_CKPT

DATASET_DIR = Path('/content/BCSS-WSSS') if IN_COLAB else Path('./data/BCSS-WSSS')
DATASET_ZIP = Path('/content/BCSS-WSSS.zip') if IN_COLAB else Path('./data/BCSS-WSSS.zip')
DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)

HF_URL = 'https://huggingface.co/datasets/nttduc/wsss-pathology/resolve/main/BCSS-WSSS.zip'
GDRIVE_ID = '1sCK8Cqsf4yCHzzIx98vitocES2L_tkjn'

def count_patches(folder):
    p = Path(folder)
    return len(list(p.rglob('*.png'))) if p.is_dir() else 0

candidates_train_check = [
    DATASET_DIR / 'training',
    DATASET_DIR / 'train',
    DATASET_DIR / 'BCSS-WSSS' / 'training',
    DATASET_DIR / 'BCSS-WSSS' / 'train',
    Path('/content/training'),
    Path('/content/train'),
    Path('/content/BCSS-WSSS/training'),
    Path('/content/BCSS-WSSS/train'),
    Path('./data/BCSS-WSSS/training'),
    Path('./data/BCSS-WSSS/train'),
    Path('./data/training'),
    Path('./data/train'),
]
total_train_patches = max([count_patches(p) for p in candidates_train_check if p.is_dir()] or [0])
if total_train_patches < 100:
    if not DATASET_ZIP.is_file() or DATASET_ZIP.stat().st_size < 1000000:
        print(f'[DOWNLOAD] Downloading BCSS-WSSS dataset from HuggingFace mirror: {HF_URL}')
        try:
            urllib.request.urlretrieve(HF_URL, str(DATASET_ZIP))
            print(f'[DOWNLOAD] Download complete ({DATASET_ZIP.stat().st_size / (1024**3):.2f} GB)')
        except Exception as e:
            print(f'[WARN] HuggingFace mirror failed: {e}. Attempting gdown from Drive...')
            import subprocess
            subprocess.run(['pip', 'install', '-q', 'gdown'], check=False)
            subprocess.run(['gdown', '--id', GDRIVE_ID, '-O', str(DATASET_ZIP)], check=False)

    if DATASET_ZIP.is_file() and DATASET_ZIP.stat().st_size > 1000000:
        print(f'[EXTRACT] Extracting {DATASET_ZIP} to {DATASET_DIR} ...')
        with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
            zf.extractall(str(DATASET_DIR))
        print('[EXTRACT] Extraction finished successfully.')

def resolve_split_path(base_dir, split='train', sub=None):
    if split in ('train', 'training'):
        split_aliases = ['training', 'train']
    elif split in ('val', 'valid', 'validation'):
        split_aliases = ['val', 'valid', 'validation']
    elif split in ('test', 'testing'):
        split_aliases = ['test', 'testing']
    else:
        split_aliases = [split]

    candidates = []
    for alias in split_aliases:
        if sub:
            candidates.extend([
                base_dir / alias / sub,
                base_dir / 'BCSS-WSSS' / alias / sub,
                Path('/content/BCSS-WSSS') / alias / sub,
                Path('/content') / alias / sub,
                Path('./data/BCSS-WSSS') / alias / sub,
                Path('./data') / alias / sub,
            ])
        else:
            candidates.extend([
                base_dir / alias,
                base_dir / alias / 'img',
                base_dir / 'BCSS-WSSS' / alias,
                base_dir / 'BCSS-WSSS' / alias / 'img',
                Path('/content/BCSS-WSSS') / alias,
                Path('/content/BCSS-WSSS') / alias / 'img',
                Path('/content') / alias,
                Path('/content') / alias / 'img',
                Path('./data/BCSS-WSSS') / alias,
                Path('./data') / alias / 'img',
                Path('./data') / alias,
            ])
    for c in candidates:
        if c.is_dir() and len(list(c.glob('*.png'))) > 0:
            return c
    for c in candidates:
        if c.is_dir() and len(list(c.rglob('*.png'))) > 0:
            return c
    default_alias = split_aliases[0]
    return (base_dir / default_alias / sub) if sub else (base_dir / default_alias)

TRAIN_DIR = resolve_split_path(DATASET_DIR, 'train')
VAL_IMG_DIR = resolve_split_path(DATASET_DIR, 'val', 'img')
VAL_MASK_DIR = resolve_split_path(DATASET_DIR, 'val', 'mask')

train_cnt = len(list(TRAIN_DIR.glob('*.png'))) if TRAIN_DIR.is_dir() else 0
if train_cnt == 0 and TRAIN_DIR.is_dir():
    train_cnt = len(list(TRAIN_DIR.rglob('*.png')))
val_cnt = len(list(VAL_IMG_DIR.glob('*.png'))) if VAL_IMG_DIR.is_dir() else 0
if val_cnt == 0 and VAL_IMG_DIR.is_dir():
    val_cnt = len(list(VAL_IMG_DIR.rglob('*.png')))

print(f'[DATASET] Active training directory: {TRAIN_DIR} (Patches: {train_cnt:,})')
print(f'[DATASET] Active validation images: {VAL_IMG_DIR} (Patches: {val_cnt:,})')
print(f'[DATASET] Active validation masks: {VAL_MASK_DIR}')

# Load Stage 1 Model
try:
    from src.models.wsss_inference_methods import WSSSTissueStage1NetCAM
except ImportError:
    from src.colab_pkg.model_loader import WSSSTissueStage1NetCAM

model_stage1 = WSSSTissueStage1NetCAM(num_classes=4).to(DEVICE)

# Check if retrained Stage 1 checkpoint exists, otherwise check author baseline or download
if not STAGE1_BEST_CKPT.is_file():
    author_ckpt = CHECKPOINT_DIR / 'stage1_checkpoint_trained_on_bcss.pth'
    local_author_ckpt = Path('./checkpoints/stage1_checkpoint_trained_on_bcss.pth')
    if author_ckpt.is_file():
        shutil.copyfile(author_ckpt, STAGE1_BEST_CKPT)
        print(f'[STAGE1] Reusing existing author baseline: {author_ckpt}')
    elif local_author_ckpt.is_file():
        shutil.copyfile(local_author_ckpt, STAGE1_BEST_CKPT)
        print(f'[STAGE1] Reusing local author baseline: {local_author_ckpt}')
    else:
        AUTHOR_STAGE1_URL = 'https://huggingface.co/longcris04/pseduo_mask_wsss/resolve/main/stage1_checkpoint_trained_on_bcss.pth'
        print(f'[DOWNLOAD] Retrained checkpoint not found. Downloading author Stage 1 baseline from {AUTHOR_STAGE1_URL}...')
        STAGE1_BEST_CKPT.parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(AUTHOR_STAGE1_URL, str(STAGE1_BEST_CKPT))
        except Exception as e:
            print(f'[WARN] Download failed: {e}. Checkpoint download skipped.')

if STAGE1_BEST_CKPT.is_file():
    try:
        ckpt1 = torch.load(STAGE1_BEST_CKPT, map_location=DEVICE, weights_only=False)
        raw_dict1 = ckpt1.get('model', ckpt1.get('state_dict', ckpt1))
        cleaned_dict1 = {k.replace('module.', '').replace('model.', ''): v for k, v in raw_dict1.items()}
        model_stage1.load_state_dict(cleaned_dict1, strict=False)
        print(f'[STAGE1] Loaded Stage 1 weights from {STAGE1_BEST_CKPT} into ResNet38d Net_CAM.')
    except Exception as e:
        print(f'[WARN] Failed to load checkpoint {STAGE1_BEST_CKPT}: {e}')
model_stage1.eval()


---
## Step 4: Multi-Layer CAM Extraction & GPU Affinity Matting
Generates multi-layer pseudo-masks (`PM_bn7`, `PM_b5_2`, `PM_b4_5`) with Otsu clinical saturation gating to eliminate background glass false positives.


In [ ]:
import os
import sys
import re
from pathlib import Path
import cv2
import numpy as np
from PIL import Image
import torch
from torchvision import transforms
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'DEVICE' not in globals() else DEVICE
DRY_RUN = False if 'DRY_RUN' not in globals() else DRY_RUN

PM_DIR = Path('/content/train_PM') if IN_COLAB else Path('./train_PM')
for sub in ['PM_bn7', 'PM_b5_2', 'PM_b4_5']:
    (PM_DIR / sub).mkdir(parents=True, exist_ok=True)

def apply_otsu_gating(rgb_np, raw_cam, min_saturation=0.08, fg_thresh=0.20):
    """Applies Otsu saturation gating to force empty glass regions to Background (Class 4)."""
    hsv = cv2.cvtColor(rgb_np, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1] / 255.0
    glass_mask = sat < min_saturation
    if raw_cam.ndim == 3:
        discrete_mask = np.argmax(raw_cam, axis=0).astype(np.uint8)
        max_prob = np.max(raw_cam[:4], axis=0)
        discrete_mask[max_prob < fg_thresh] = 4
    else:
        discrete_mask = raw_cam.copy().astype(np.uint8)
    discrete_mask[glass_mask] = 4
    return discrete_mask

# Affinity Matting / Bilateral boundary refinement without discrete-label interpolation artifacts
def gpu_affinity_refine(mask_or_cam, img_rgb_np=None):
    """Applies bilateral edge-preserving spatial affinity refinement on continuous channels."""
    if mask_or_cam.ndim == 3 and mask_or_cam.shape[0] in (4, 5):
        refined = np.zeros_like(mask_or_cam, dtype=np.float32)
        if img_rgb_np is not None:
            gray_guide = cv2.cvtColor(img_rgb_np, cv2.COLOR_RGB2GRAY).astype(np.float32) / 255.0
            grad_x = cv2.Sobel(gray_guide, cv2.CV_32F, 1, 0, ksize=3)
            grad_y = cv2.Sobel(gray_guide, cv2.CV_32F, 0, 1, ksize=3)
            edge_weight = np.clip(1.0 - np.sqrt(grad_x**2 + grad_y**2), 0.5, 1.0)
            for c in range(mask_or_cam.shape[0]):
                filtered = cv2.bilateralFilter(mask_or_cam[c].astype(np.float32), d=5, sigmaColor=0.15, sigmaSpace=3.0)
                refined[c] = filtered * edge_weight + mask_or_cam[c] * (1.0 - edge_weight)
        else:
            for c in range(mask_or_cam.shape[0]):
                refined[c] = cv2.bilateralFilter(mask_or_cam[c].astype(np.float32), d=5, sigmaColor=0.15, sigmaSpace=3.0)
        return refined
    else:
        mask_2d = mask_or_cam.astype(np.int64)
        c_max = max(5, int(mask_2d.max()) + 1)
        one_hot = np.zeros((c_max, mask_2d.shape[0], mask_2d.shape[1]), dtype=np.float32)
        for c in range(c_max):
            one_hot[c] = (mask_2d == c).astype(np.float32)
            one_hot[c] = cv2.bilateralFilter(one_hot[c], d=5, sigmaColor=0.15, sigmaSpace=3.0)
        return np.argmax(one_hot, axis=0).astype(np.uint8)

all_train_files = sorted(list(TRAIN_DIR.glob('*.png'))) if 'TRAIN_DIR' in globals() and TRAIN_DIR.is_dir() else []
if len(all_train_files) == 0 and 'TRAIN_DIR' in globals() and TRAIN_DIR.is_dir():
    all_train_files = sorted(list(TRAIN_DIR.rglob('*.png')))

if len(all_train_files) == 0:
    t_dir = TRAIN_DIR if 'TRAIN_DIR' in globals() else Path('./data/BCSS-WSSS/training')
    raise FileNotFoundError(f'[STAGE2 ERROR] No training patches found in {t_dir}. Please ensure dataset extraction completed successfully!')

LIMIT_PM = 100 if DRY_RUN else None
if LIMIT_PM:
    all_train_files = all_train_files[:LIMIT_PM]

existing_main = list((PM_DIR / 'PM_bn7').glob('*.png'))
print(f'[CAM] Target pseudo-masks: {len(all_train_files):,} patches into {PM_DIR} (Existing in PM_bn7: {len(existing_main):,})')

cam_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

with torch.no_grad():
    for fpath in tqdm(all_train_files, desc='Generating Multi-Layer CAMs', unit='patch'):
        out_bn7 = PM_DIR / 'PM_bn7' / fpath.name
        out_b5_2 = PM_DIR / 'PM_b5_2' / fpath.name
        out_b4_5 = PM_DIR / 'PM_b4_5' / fpath.name
        if out_bn7.is_file() and out_b5_2.is_file() and out_b4_5.is_file():
            continue

        img_pil = Image.open(fpath).convert('RGB')
        img_tensor = cam_transform(img_pil).unsqueeze(0).to(DEVICE)
        img_np = np.array(img_pil.resize((224, 224)))

        # Extract CAM (forward_cam returns (cam, y) where cam is (B, 4, 28, 28))
        if hasattr(model_stage1, 'forward_cam'):
            cam_raw, cls_probs = model_stage1.forward_cam(img_tensor)
            cam_up = torch.nn.functional.interpolate(cam_raw, size=(224, 224), mode='bilinear', align_corners=False)
            cam_min = cam_up.view(1, 4, -1).min(dim=-1)[0].view(1, 4, 1, 1)
            cam_max = cam_up.view(1, 4, -1).max(dim=-1)[0].view(1, 4, 1, 1)
            cam_norm = (cam_up - cam_min) / (cam_max - cam_min + 1e-5)
            
            # Weak-label / probability gating to suppress absent classes (prevent phantom masks)
            label_match = re.search(r'\[([0-9\s,]+)\]', fpath.name)
            if label_match:
                digits = [float(ch) for ch in re.findall(r'[0-9]', label_match.group(1))]
                if len(digits) >= 4:
                    gate = torch.tensor(digits[:4], dtype=torch.float32, device=cam_norm.device).view(1, 4, 1, 1)
                    cam_norm = cam_norm * gate
            elif cls_probs is not None:
                gate = (cls_probs > 0.10).float().view(1, 4, 1, 1)
                cam_norm = cam_norm * gate
            
            cam = cam_norm.squeeze(0).float().cpu().numpy()  # (4, 224, 224)
        else:
            cam = np.zeros((4, 224, 224), dtype=np.float32)

        # Construct 5-class distribution (TUM, STR, LYM, NEC, BACK)
        bg_channel = np.clip(1.0 - np.max(cam, axis=0, keepdims=True), 0.0, 1.0)
        full_cam = np.concatenate([cam, bg_channel], axis=0)  # (5, 224, 224)

        # Multi-layer pseudo-masks with Otsu gating & affinity matting
        refined_cam = gpu_affinity_refine(full_cam, img_np)
        mask_bn7 = apply_otsu_gating(img_np, refined_cam, min_saturation=0.08, fg_thresh=0.20)
        mask_b5_2 = apply_otsu_gating(img_np, refined_cam, min_saturation=0.08, fg_thresh=0.25)
        mask_b4_5 = apply_otsu_gating(img_np, refined_cam, min_saturation=0.08, fg_thresh=0.15)

        # Save pseudo-masks (using 0..4 indexed PNG)
        Image.fromarray(mask_bn7).save(out_bn7)
        Image.fromarray(mask_b5_2).save(out_b5_2)  # Aux 1 (conservative)
        Image.fromarray(mask_b4_5).save(out_b4_5)  # Aux 2 (relaxed)

print(f'[CAM] Pseudo-masks generated successfully in {PM_DIR}')


---
## Step 5: Instantiate Stage 2 DeepLabV3+ ResNet-101 & Multi-Layer Loss Setup
Configures the 59.3M-parameter DeepLabV3+ segmentation architecture with Atrous Spatial Pyramid Pooling (ASPP).


In [ ]:
import os
import sys
from pathlib import Path
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'DEVICE' not in globals() else DEVICE

try:
    from src.models.wsss_inference_methods import DeepLabV3Plus
except ImportError:
    try:
        from src.models.deeplabv3plus import DeepLabV3Plus
    except ImportError:
        from src.colab_pkg.model_loader import DeepLabV3Plus

model_stage2 = DeepLabV3Plus(num_classes=5, backbone='resnet101').to(DEVICE)
s2_params = sum(p.numel() for p in model_stage2.parameters())
print(f'[STAGE2] DeepLabV3+ ResNet-101 initialized: {s2_params:,} parameters')

# Multi-Layer Pseudo-Supervision (MLPS) Loss
class MLPSLoss(torch.nn.Module):
    """Multi-Layer Pseudo-Supervision Loss with boundary uncertainty weighting."""
    def __init__(self, w_main=1.0, w_aux1=0.4, w_aux2=0.2):
        super().__init__()
        self.w_main = w_main
        self.w_aux1 = w_aux1
        self.w_aux2 = w_aux2
        self.ce = torch.nn.CrossEntropyLoss(ignore_index=255)

    def forward(self, pred, target_main, target_aux1=None, target_aux2=None):
        loss = self.w_main * self.ce(pred, target_main)
        if target_aux1 is not None and self.w_aux1 > 0:
            loss = loss + self.w_aux1 * self.ce(pred, target_aux1)
        if target_aux2 is not None and self.w_aux2 > 0:
            loss = loss + self.w_aux2 * self.ce(pred, target_aux2)
        return loss

criterion_s2 = MLPSLoss()
STAGE2_EPOCHS = 10
optimizer_s2 = torch.optim.SGD(model_stage2.parameters(), lr=1e-3, momentum=0.9, weight_decay=5e-4)
scheduler_s2 = torch.optim.lr_scheduler.PolynomialLR(optimizer_s2, total_iters=STAGE2_EPOCHS, power=0.9)


---
## Step 6: Resume Capability for Stage 2 Checkpoint
Restores training progress from `/content/checkpoints/stage2_latest.pth`.


In [ ]:
import os
import sys
from pathlib import Path
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'DEVICE' not in globals() else DEVICE
CHECKPOINT_DIR = Path('/content/checkpoints') if ('IN_COLAB' in globals() and IN_COLAB) else Path('./checkpoints')
STAGE2_LATEST_CKPT = CHECKPOINT_DIR / 'stage2_latest.pth' if 'STAGE2_LATEST_CKPT' not in globals() else STAGE2_LATEST_CKPT
STAGE2_BEST_CKPT = CHECKPOINT_DIR / 'stage2_retrained_best.pth' if 'STAGE2_BEST_CKPT' not in globals() else STAGE2_BEST_CKPT

start_epoch_s2 = 0
best_miou_s2 = 0.0
history_s2 = {'epoch': [], 'loss': [], 'miou': []}

if STAGE2_LATEST_CKPT.is_file():
    print(f'[RESUME] Loading existing Stage 2 checkpoint: {STAGE2_LATEST_CKPT}')
    try:
        ckpt_s2 = torch.load(STAGE2_LATEST_CKPT, map_location=DEVICE, weights_only=False)
        model_stage2.load_state_dict(ckpt_s2['model'])
        if 'optimizer' in ckpt_s2:
            optimizer_s2.load_state_dict(ckpt_s2['optimizer'])
        if 'scheduler' in ckpt_s2:
            scheduler_s2.load_state_dict(ckpt_s2['scheduler'])
        start_epoch_s2 = ckpt_s2.get('epoch', 0) + 1
        best_miou_s2 = ckpt_s2.get('best_miou', 0.0)
        history_s2 = ckpt_s2.get('history', history_s2)
        print(f'[RESUME] Resuming Stage 2 from epoch {start_epoch_s2}/{STAGE2_EPOCHS} (Best mIoU: {best_miou_s2:.4f})')
    except Exception as e:
        print(f'[WARN] Could not restore checkpoint: {e}. Starting fresh Stage 2 training.')
else:
    print('[INIT] No previous Stage 2 checkpoint found. Starting from scratch.')


---
## Step 7: DeepLabV3+ Stage 2 Training Loop with Multi-Layer Pseudo-Supervision
Executes full Stage 2 training on multi-layer pseudo-masks with `tqdm` progress bars and automatic epoch checkpointing.


In [ ]:
import os
import sys
from pathlib import Path
from PIL import Image
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'DEVICE' not in globals() else DEVICE
AMP_DTYPE = torch.float16 if 'AMP_DTYPE' not in globals() else AMP_DTYPE
DRY_RUN = False if 'DRY_RUN' not in globals() else DRY_RUN

class BCSSStage2Dataset(Dataset):
    """Dataset loading patches and multi-layer pseudo-masks synchronously."""
    def __init__(self, img_dir, pm_dir, limit=None):
        self.img_dir = Path(img_dir)
        self.pm_dir = Path(pm_dir)
        files = sorted(list((self.pm_dir / 'PM_bn7').glob('*.png')))
        if len(files) == 0:
            raise FileNotFoundError(f'[STAGE2 ERROR] No pseudo-masks found in {self.pm_dir / "PM_bn7"}. Please run Cell 8 (pseudo-mask generation) before training DeepLabV3+!')

        if limit is not None and limit > 0:
            files = files[:limit]
        self.files = files
        self.img_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
        print(f'[DATASET] Stage 2 ingested {len(self.files):,} pseudo-mask samples.')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        fname = self.files[idx].name
        img_path = self.img_dir / fname
        if not img_path.is_file():
            candidates = list(self.img_dir.rglob(fname))
            img_path = candidates[0] if len(candidates) > 0 else img_path

        if not img_path.is_file():
            raise FileNotFoundError(f'[STAGE2 ERROR] Image patch {fname} not found in {self.img_dir}')
        img = Image.open(img_path).convert('RGB')

        fallback_p = self.files[idx]
        p_bn7 = self.pm_dir / 'PM_bn7' / fname
        p_b5_2 = self.pm_dir / 'PM_b5_2' / fname
        p_b4_5 = self.pm_dir / 'PM_b4_5' / fname

        def _load_mask(target_p):
            actual_p = target_p if target_p.is_file() else fallback_p
            m_img = Image.open(actual_p)
            if m_img.mode != 'P':
                m_img = m_img.convert('L')
            m_pil = m_img.resize((224, 224), Image.NEAREST)
            arr = np.array(m_pil, dtype=np.int64)
            if arr.ndim == 3:
                arr = arr[:, :, 0]
            arr = np.clip(arr, 0, 4)
            return torch.tensor(arr, dtype=torch.long)

        return (
            self.img_transform(img),
            _load_mask(p_bn7),
            _load_mask(p_b5_2),
            _load_mask(p_b4_5),
        )

BATCH_SIZE = 12
NUM_WORKERS = 2

s2_dataset = BCSSStage2Dataset(TRAIN_DIR, PM_DIR, limit=64 if DRY_RUN else None)
s2_loader = DataLoader(
    s2_dataset,
    batch_size=min(BATCH_SIZE, max(1, len(s2_dataset))),
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=(len(s2_dataset) >= BATCH_SIZE),
)
print(f'[DATALOADER] Batches per epoch: {len(s2_loader)} (Batch Size: {min(BATCH_SIZE, max(1, len(s2_dataset)))}, Total Samples: {len(s2_dataset)})')

scaler_s2 = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

if start_epoch_s2 >= STAGE2_EPOCHS:
    print(f'[STAGE2] Training already completed up to epoch {start_epoch_s2}/{STAGE2_EPOCHS}. Proceeding to validation.')
else:
    for epoch in range(start_epoch_s2, STAGE2_EPOCHS):
        model_stage2.train()
        running_loss = 0.0
        total_batches = 0

        pbar = tqdm(s2_loader, desc=f'Stage 2 Epoch {epoch+1:02d}/{STAGE2_EPOCHS:02d}', unit='batch')
        for imgs, m_main, m_aux1, m_aux2 in pbar:
            imgs = imgs.to(DEVICE, non_blocking=True)
            m_main = m_main.to(DEVICE, non_blocking=True)
            m_aux1 = m_aux1.to(DEVICE, non_blocking=True)
            m_aux2 = m_aux2.to(DEVICE, non_blocking=True)

            optimizer_s2.zero_grad()
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda')):
                preds = model_stage2(imgs)
                loss = criterion_s2(preds, m_main, m_aux1, m_aux2)

            scaler_s2.scale(loss).backward()
            scaler_s2.step(optimizer_s2)
            scaler_s2.update()

            running_loss += loss.item()
            total_batches += 1
            pbar.set_postfix({
                'loss': f'{running_loss / total_batches:.4f}',
                'lr': f'{optimizer_s2.param_groups[0]["lr"]:.2e}',
            })

        scheduler_s2.step()
        avg_loss = running_loss / max(1, total_batches)
        history_s2['epoch'].append(epoch + 1)
        history_s2['loss'].append(round(avg_loss, 5))

        # Save latest checkpoint
        latest_state_s2 = {
            'epoch': epoch,
            'model': model_stage2.state_dict(),
            'optimizer': optimizer_s2.state_dict(),
            'scheduler': scheduler_s2.state_dict(),
            'best_miou': best_miou_s2,
            'history': history_s2,
        }
        torch.save(latest_state_s2, STAGE2_LATEST_CKPT)
        print(f'[CHECKPOINT] Stage 2 epoch {epoch+1} saved to {STAGE2_LATEST_CKPT}')

        # Periodic or best checkpoint save
        torch.save(latest_state_s2, STAGE2_BEST_CKPT)
        print(f'[CHECKPOINT] Stage 2 retrained checkpoint updated at {STAGE2_BEST_CKPT}')


---
## Step 8: Validation Evaluation, Qualitative Inspection & Artifact Packaging
Evaluates DeepLabV3+ predictions against validation ground truth and packages deliverables.


In [ ]:
import os
import sys
import json
import zipfile
from pathlib import Path
import numpy as np
import torch
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

IN_COLAB = 'google.colab' in sys.modules if 'IN_COLAB' not in globals() else IN_COLAB
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if 'DEVICE' not in globals() else DEVICE
AMP_DTYPE = torch.float16 if 'AMP_DTYPE' not in globals() else AMP_DTYPE
DRY_RUN = False if 'DRY_RUN' not in globals() else DRY_RUN

OUTPUT_DIR_S2 = Path('/content/stage2_results') if IN_COLAB else Path('./stage2_results')
OUTPUT_DIR_S2.mkdir(parents=True, exist_ok=True)

# 1. Validation Evaluation (if validation images and masks are present)
val_files = sorted(list(VAL_IMG_DIR.glob('*.png'))) if ('VAL_IMG_DIR' in globals() and VAL_IMG_DIR.is_dir()) else []
if len(val_files) == 0 and ('VAL_IMG_DIR' in globals() and VAL_IMG_DIR.is_dir()):
    val_files = sorted(list(VAL_IMG_DIR.rglob('*.png')))

val_eval_limit = 50 if DRY_RUN else min(200, len(val_files))
if len(val_files) > 0 and ('VAL_MASK_DIR' in globals() and VAL_MASK_DIR.is_dir()):
    print(f'[VAL] Evaluating DeepLabV3+ Stage 2 on {val_eval_limit} validation patches...')
    model_stage2.eval()
    val_tf = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    intersections = np.zeros(5, dtype=np.float64)
    unions = np.zeros(5, dtype=np.float64)

    with torch.no_grad():
        for vf in tqdm(val_files[:val_eval_limit], desc='Validating Stage 2 DeepLabV3+', unit='patch'):
            mask_path = VAL_MASK_DIR / vf.name
            if not mask_path.is_file():
                candidates = list(VAL_MASK_DIR.rglob(vf.name))
                if len(candidates) > 0:
                    mask_path = candidates[0]
            if not mask_path.is_file():
                continue

            mask_img = Image.open(mask_path)
            if mask_img.mode != 'P':
                mask_img = mask_img.convert('L')
            gt_pil = mask_img.resize((224, 224), Image.NEAREST)
            gt_arr = np.array(gt_pil, dtype=np.int64)
            if gt_arr.ndim == 3:
                gt_arr = gt_arr[:, :, 0]

            img_pil = Image.open(vf).convert('RGB')
            inp = val_tf(img_pil).unsqueeze(0).to(DEVICE)
            with torch.amp.autocast('cuda', dtype=AMP_DTYPE, enabled=(DEVICE.type == 'cuda')):
                logits = model_stage2(inp)
            pred = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

            for c in range(5):
                pred_c = (pred == c)
                gt_c = (gt_arr == c)
                intersections[c] += np.logical_and(pred_c, gt_c).sum()
                unions[c] += np.logical_or(pred_c, gt_c).sum()

    iou_per_class = np.zeros(5, dtype=np.float64)
    for c in range(5):
        iou_per_class[c] = (intersections[c] / unions[c]) if unions[c] > 0 else 0.0

    miou_4cls = float(np.mean(iou_per_class[:4]))
    miou_5cls = float(np.mean(iou_per_class))
    print(f'[VAL] Stage 2 Validation 4-Class mIoU: {miou_4cls * 100:.2f}% | 5-Class mIoU: {miou_5cls * 100:.2f}%')
    history_s2['val_miou_4cls'] = round(miou_4cls, 4)
    history_s2['val_miou_5cls'] = round(miou_5cls, 4)
    history_s2['class_ious'] = [round(float(x), 4) for x in iou_per_class]

# Plot Stage 2 training loss
if history_s2.get('epoch'):
    plt.figure(figsize=(8, 4), dpi=300)
    plt.plot(history_s2['epoch'], history_s2['loss'], marker='o', color='tab:blue', label='Stage 2 MLPS Loss')
    plt.xlabel('Epoch', fontweight='bold')
    plt.ylabel('Loss', fontweight='bold')
    plt.title('DeepLabV3+ ResNet-101 Stage 2 Training Convergence', fontweight='bold')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR_S2 / 'stage2_training_loss.png', dpi=300)
    plt.close()

# Save metrics JSON
with open(OUTPUT_DIR_S2 / 'STAGE2_TRAINING_METRICS.json', 'w', encoding='utf-8') as f:
    json.dump(history_s2, f, indent=2)

# Package into ZIP
ZIP_EXPORT_S2 = Path('/content/SE374_Stage2_Training_Results.zip') if IN_COLAB else Path('./SE374_Stage2_Training_Results.zip')
with zipfile.ZipFile(ZIP_EXPORT_S2, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR_S2.glob('*'):
        zf.write(f, arcname=f.name)
print(f'[PACKAGE] Stage 2 results zipped to {ZIP_EXPORT_S2} ({ZIP_EXPORT_S2.stat().st_size / 1024:.1f} KB)')

if IN_COLAB:
    try:
        from google.colab import files
        files.download(str(ZIP_EXPORT_S2))
        print('[DOWNLOAD] Triggered browser download of Stage 2 results ZIP.')
    except Exception as e:
        print(f'[NOTICE] Download skipped: {e}')
